In [ ]:
import os, subprocess, sys

SENTINEL = "/content/.canary_setup_done"

if not os.path.exists(SENTINEL):
    def sh(c):
        print("$", c); subprocess.run(c, shell=True, check=False)

    print(">>> PHASE 1: installing dependencies (one-time)...\n")
    sh("apt-get -qq update")
    sh("apt-get -qq install -y libsndfile1 ffmpeg > /dev/null")
    sh('pip install -q "nemo_toolkit[asr]"')
    sh("pip install -q librosa soundfile pydub")
    sh('pip install -q --force-reinstall --no-cache-dir "numpy>=2.2,<2.4" "scipy>=1.15"')

    open(SENTINEL, "w").write("done")
    print("\n✅ Setup complete. Restarting the runtime now.")
    print("   When it reconnects, RUN THIS CELL AGAIN to start the tutorial.")
    os.kill(os.getpid(), 9)

In [ ]:
import time, json, gc, math, urllib.request
import torch, numpy as np, soundfile as sf, librosa

print(">>> PHASE 2: running tutorial\n")
print("NumPy:", np.__version__, "| PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0),
          f"| VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
else:
    print("⚠️  No GPU — will run on CPU (very slow). "
          "Set Runtime > Change runtime type > GPU.")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

LANGS = {
    "bg":"Bulgarian","hr":"Croatian","cs":"Czech","da":"Danish","nl":"Dutch",
    "en":"English","et":"Estonian","fi":"Finnish","fr":"French","de":"German",
    "el":"Greek","hu":"Hungarian","it":"Italian","lv":"Latvian","lt":"Lithuanian",
    "mt":"Maltese","pl":"Polish","pt":"Portuguese","ro":"Romanian","sk":"Slovak",
    "sl":"Slovenian","es":"Spanish","sv":"Swedish","ru":"Russian","uk":"Ukrainian",
}
print(f"\nSupported languages ({len(LANGS)}):", ", ".join(LANGS.keys()))

from nemo.collections.asr.models import ASRModel
print("\nLoading nvidia/canary-1b-v2 ...")
t0 = time.time()
asr_model = ASRModel.from_pretrained(model_name="nvidia/canary-1b-v2").to(DEVICE).eval()
print(f"Model loaded in {time.time()-t0:.1f}s")

In [ ]:
TARGET_SR = 16000
def prepare_audio(path_or_url, out_path=None):
    if str(path_or_url).startswith(("http://", "https://")):
        local = "/content/_dl_" + os.path.basename(path_or_url.split("?")[0])
        urllib.request.urlretrieve(path_or_url, local)
        path_or_url = local
    audio, _ = librosa.load(path_or_url, sr=TARGET_SR, mono=True)
    if out_path is None:
        base = os.path.splitext(os.path.basename(path_or_url))[0]
        out_path = f"/content/{base}_16k_mono.wav"
    sf.write(out_path, audio, TARGET_SR, subtype="PCM_16")
    dur = len(audio) / TARGET_SR
    print(f"Prepared: {out_path}  ({dur:.1f}s, 16kHz, mono)")
    return out_path, dur

SAMPLE_URL = "https://dldata-public.s3.us-east-2.amazonaws.com/2086-149220-0033.wav"
sample_wav, sample_dur = prepare_audio(SAMPLE_URL)

def transcribe(files, source_lang="en", target_lang="en", timestamps=False, batch_size=1):
    if isinstance(files, str):
        files = [files]
    return asr_model.transcribe(files, source_lang=source_lang, target_lang=target_lang,
                                timestamps=timestamps, batch_size=batch_size)

print("\n=== 1) BASIC ASR (English) ===")
res = transcribe(sample_wav, source_lang="en", target_lang="en")
print("Transcript:", res[0].text)

print("\n=== 2) TRANSLATION (EN audio -> X) ===")
for tgt in ["fr", "de", "es", "it"]:
    out = transcribe(sample_wav, source_lang="en", target_lang=tgt)
    print(f"  EN -> {LANGS[tgt]:<10} ({tgt}): {out[0].text}")

In [ ]:
print("\n=== 3) TIMESTAMPS (ASR) ===")
ts_out = transcribe(sample_wav, source_lang="en", target_lang="en", timestamps=True)
word_ts = ts_out[0].timestamp.get("word", [])
seg_ts  = ts_out[0].timestamp.get("segment", [])
print("Segments:")
for s in seg_ts:
    print(f"  [{s['start']:6.2f}s - {s['end']:6.2f}s]  {s['segment']}")
print("First 10 words:")
for w in word_ts[:10]:
    print(f"  [{w['start']:6.2f}s - {w['end']:6.2f}s]  {w['word']}")

def _srt_time(t):
    h=int(t//3600); m=int((t%3600)//60); s=int(t%60); ms=int(round((t-int(t))*1000))
    return f"{h:02d}:{m:02d}:{s:02d},{ms:03d}"

def segments_to_srt(segments, out_path="/content/output.srt"):
    lines=[]
    for i, seg in enumerate(segments, 1):
        lines += [str(i), f"{_srt_time(seg['start'])} --> {_srt_time(seg['end'])}",
                  seg["segment"].strip(), ""]
    open(out_path, "w", encoding="utf-8").write("\n".join(lines))
    print(f"Saved SRT: {out_path}")
    return out_path

print("\n=== 4) SRT EXPORT (translated French subtitles) ===")
fr_ts = transcribe(sample_wav, source_lang="en", target_lang="fr", timestamps=True)
segments_to_srt(fr_ts[0].timestamp["segment"], "/content/subtitles_fr.srt")
print(open("/content/subtitles_fr.srt").read())

In [1]:
print("\n=== 5) LONG-FORM (sample tiled x6) ===")
long_audio, _ = librosa.load(sample_wav, sr=TARGET_SR, mono=True)
long_audio = np.tile(long_audio, 6)
sf.write("/content/long.wav", long_audio, TARGET_SR, subtype="PCM_16")
print(f"Long clip duration: {len(long_audio)/TARGET_SR:.1f}s")
long_out = transcribe("/content/long.wav", source_lang="en", target_lang="en", batch_size=1)
print("Long transcript (first 300 chars):", long_out[0].text[:300], "...")

print("\n=== 6) BATCH ===")
for name in ["clip_a", "clip_b"]:
    sf.write(f"/content/{name}.wav",
             librosa.load(sample_wav, sr=TARGET_SR, mono=True)[0], TARGET_SR, subtype="PCM_16")
batch = transcribe(["/content/clip_a.wav", "/content/clip_b.wav"],
                   source_lang="en", target_lang="en", batch_size=2)
for i, b in enumerate(batch):
    print(f"  file {i}: {b.text}")

print("\n=== 7) BENCHMARK ===")
t0 = time.time(); _ = transcribe(sample_wav, source_lang="en", target_lang="en")
elapsed = time.time()-t0
print(f"Audio: {sample_dur:.2f}s | Compute: {elapsed:.2f}s | RTFx ≈ {sample_dur/elapsed:.1f}x")

print("\n✅ Done. Change source_lang/target_lang from the LANGS dict to try other languages.")

>>> PHASE 2: running tutorial

NumPy: 2.3.5 | PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4 | VRAM: 15.6 GB

Supported languages (25): bg, hr, cs, da, nl, en, et, fi, fr, de, el, hu, it, lv, lt, mt, pl, pt, ro, sk, sl, es, sv, ru, uk


[NeMo W 2026-06-11 08:29:33 megatron_init:62] Megatron num_microbatches_calculator not found, using Apex version.
[NeMo W 2026-06-11 08:29:36 nemo_logging:364] /usr/local/lib/python3.12/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
      m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
    
[NeMo W 2026-06-11 08:29:36 nemo_logging:364] /usr/local/lib/python3.12/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
      m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
    
[NeMo W 2026-06-11 08:29:36 nemo_logging:364] /usr/local/lib/python3.12/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
      elif re.match('(flt)p?( \(default\))?$', token):
    
[NeMo W 2026-06-11 08:29:36 nemo_logging:364] /usr/local/lib/python3.12/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
      elif re.match('(dbl)p?( \(default\))?$', token):
    



Loading nvidia/canary-1b-v2 ...


canary-1b-v2.nemo:   0%|          | 0.00/6.36G [00:00<?, ?B/s]

[NeMo I 2026-06-11 08:31:33 mixins:184] Tokenizer CanaryBPETokenizer initialized with 16384 tokens


...

Long transcript (first 300 chars): Well, I don't wish to see it any more, observed Phoebe, turning away her eyes. It is certainly very like the old portrait. Well, I don't wish to see it any more, observed Phoebe, turning away her eyes. It is certainly very like the old portrait. Well, I don't wish to see it any more, observed Phoebe ...

=== 6) BATCH ===


Transcribing: 1it [00:00,  1.80it/s]
[NeMo W 2026-06-11 08:32:58 dataloader:826] The following configuration keys are ignored by Lhotse dataloader: trim_silence,enable_chunking
[NeMo W 2026-06-11 08:32:58 dataloader:523] You are using a non-tarred dataset and requested tokenization during data sampling (pretokenize=True). This will cause the tokenization to happen in the main (GPU) process,possibly impacting the training speed if your tokenizer is very large.If the impact is noticable, set pretokenize=False in dataloader config.(note: that will disable token-per-second filtering and 2D bucketing features)


  file 0: Well, I don't wish to see it any more, observed Phoebe, turning away her eyes. It is certainly very like the old portrait.
  file 1: Well, I don't wish to see it any more, observed Phoebe, turning away her eyes. It is certainly very like the old portrait.

=== 7) BENCHMARK ===


Transcribing: 1it [00:00,  2.10it/s]

Audio: 7.43s | Compute: 0.52s | RTFx ≈ 14.2x

✅ Done. Change source_lang/target_lang from the LANGS dict to try other languages.
